# Setup

Change `/workspaces/taegis-sdk-python-workspace/taegis-magic` to wherever your taegis-magic dir is and make sure that is on the correct branch.<br><br>
After changing the directory make sure to reload the kernel!

In [ ]:
!pip install -e /Users/chris.langreo/dev/taegis/taegis-magic-dir/taegis-magic

Installing itables for the purposes of displaying DataFrames in a nicer format

In [ ]:
!pip install itables

In [ ]:
from taegis_magic.pandas.process import process_pivot_http
import pandas as pd
from itables import show


In [ ]:
%load_ext taegis_magic

In [ ]:
%taegis auth login --use-universal-authentication

Code to create aggregate process data. Querying HTTP data first, then looking for that same data in process table, to help ensure that the aggregate process data when passed into the pipe function will return HTTP data. 

In [ ]:
%%taegis events search --assign http_data
from http
EARLIEST=-30d
  | fields host_id, sensor_type
  | head 10000


In [ ]:
host_id_needed_cols = http_data[['host_id', 'sensor_type']]
sub_queries = []
cols = ['host_id', 'sensor_type']
sub_queries = []

for _, row in host_id_needed_cols.iterrows():
    row_filters = [(
        f"{col} = '{row[col]}'"
    )
    for col in cols]
    sub_queries.append( "(" + " AND ".join(row_filters) + ")" )
unique_sub_queries = list(dict.fromkeys(sub_queries))
where_clause = " OR ".join(unique_sub_queries)
full_query = f"FROM process where {where_clause} EARLIEST=-30d | aggregate count by host_id, sensor_type"

%taegis events search --cell "$full_query" --assign agg_process_events



In [ ]:
show(agg_process_events)

# Beginning of Testing

Setting log level to DEBUG to show more detail if desired

In [ ]:
# import logging
# logging.basicConfig(level=logging.DEBUG)
# logging.getLogger("taegis_magic").setLevel(logging.DEBUG)

In [ ]:
# Constants for calling pipe functions

region="charlie"
tenant_id="11063"

## Empty DataFrame as input

For an empty DataFrame as input it should just be returned back

In [ ]:
from pandas import DataFrame
empty_df = DataFrame()

ret_df = empty_df.pipe(process_pivot_http, region=region, tenant_id=tenant_id)
print(ret_df)


## All columns in DataFrame are valid

All columns in input DataFrame are part of the `HTTP_PIVOT_COLUMNS` column

In [ ]:
from taegis_magic.pandas.process import HTTP_PIVOT_COLUMNS
print(HTTP_PIVOT_COLUMNS)

In [ ]:
show(agg_process_events.pipe(process_pivot_http, region=region, tenant_id=tenant_id, earliest="5d"))


## No columns in DataFrame are valid

Will reduce `process_events` DataFrame down to 1 column and will rename it so that the name of the column is not in `HTTP_PIVOT_COLUMNS`

In [ ]:
process_events_no_valid_columns = agg_process_events[['host_id']].copy()
process_events_no_valid_columns.rename(columns={'host_id':'my_host_id'}, inplace=True)

process_events_no_valid_columns.pipe(process_pivot_http, region=region, tenant_id=tenant_id)

## Mix of valid and invalid columns in DataFrame

Will rename just column in agg_process_events so that that one column will be part of the query and the other will not. 

In [ ]:
process_events_mixed_columns = agg_process_events.copy()
process_events_mixed_columns.rename(columns={'sensor_type':'my_sensor_type'}, inplace=True)
show(process_events_mixed_columns.pipe(process_pivot_http, region=region, tenant_id=tenant_id))


## DataFrame that contains valid columns, but no data will be returned from query

The columns of the DataFrame will be valid, but the data will be made invalid by appending a string to each cell in the DataFrame. Therefore the input DataFrame, with invalid data, will be returned. 

In [ ]:
process_events_invalid_data = agg_process_events.apply(lambda col: col.astype(str) + 'a')
ret = process_events_invalid_data.pipe(process_pivot_http, region=region, tenant_id=tenant_id)
show(ret)
